# Verifying the artifact on Google Colab

Runs the pipeline on a free Colab GPU. This is the quickest independent check of the
artifact: it clones the repository, installs the pinned environment, pulls the model,
runs the unit suite, then runs the pipeline and the mutation filter for real. Nothing
here needs credentials, and nothing is written outside the session.

It covers the two jobs an ordinary laptop cannot do: fast generation (a T4 instead of
about 2.4 tokens per second on CPU) and mutation testing, because mutmut is Linux-only.

**To run it:**

1. Set the runtime: `Runtime > Change runtime type > T4 GPU`.
2. `Runtime > Run all`. The first run takes about ten minutes, most of it the model download.

For the full campaign behind Chapter 5, rather than this smoke check, use
`colab_experiments.ipynb` instead. That one runs 12 modules across 3 seeds under three
conditions and needs several GPU sessions.


In [ ]:
# 1) Confirm the GPU is attached
!nvidia-smi

In [ ]:
# 2) Clone the repository, or fast-forward an existing clone. The repository is
# public, so this needs no credentials and no Colab secret.
%cd /content
import os
REPO = "https://github.com/youthinkyoucancode/llm-testgen-thesis.git"
if not os.path.isdir("llm-testgen-thesis"):
    !git clone --depth 1 {REPO}
%cd llm-testgen-thesis
!git pull --ff-only {REPO} main


In [ ]:
# 3) Install Ollama, start the server, pull the model (same model id as the local runs)
import shutil, subprocess, time
if shutil.which("ollama") is None:
    # zstd first: the installer ships a .tar.zst bundle and Colab's image has no zstd
    !apt-get -qq update && apt-get -qq install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh
assert shutil.which("ollama"), "Ollama install failed; the installer's ERROR line above says why"
if subprocess.run(["pgrep", "-x", "ollama"], capture_output=True).returncode != 0:
    server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
!ollama pull qwen2.5-coder
!ollama list

In [ ]:
# 4) Reproducible Python environment: the same pinned versions as the local machine
# Colab's Python ships without ensurepip, so venv creation needs this apt package
!apt-get -qq install -y python3.12-venv
!python -m venv .venv
import os
assert os.path.exists(".venv/bin/pip"), "venv has no pip; the venv output above says why"
!.venv/bin/pip install -q -r requirements.lock
!.venv/bin/python -c "import pytest, coverage, mutmut, ollama, yaml; print('env ok')"

In [ ]:
# 5) Sanity: the unit suite must be green on Linux too (no model involved)
!PYTHONPATH=src .venv/bin/python -m pytest tests -q

In [ ]:
# 6) Pipeline smoke: the full loop on the fixture module, real model, GPU config.
# Expect the same shape as the local run (coverage in the 90s, stop on no-gain)
# but each round in seconds instead of minutes.
!PYTHONPATH=src .venv/bin/python -m llmtestgen.cli tests/fixtures/sample_module.py --condition C --config config/colab.yaml

In [ ]:
# 7) Mutation smoke: the first live run of filters/mutation.py.
# mutmut injects small bugs into sample_module; the human-written suite tries
# to catch them. Expect a mutant count, a score, and a few survivors.
!PYTHONPATH=src .venv/bin/python -m llmtestgen.filters.mutation tests/fixtures/sample_module.py tests/fixtures/sample_module_human_tests.py

In [ ]:
# 8) Save results off the VM (Colab machines are wiped when the session ends)
!zip -qr results.zip experiments/results
from google.colab import files
files.download("results.zip")

# Alternative: copy into Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/thesis_results && cp -r experiments/results/* /content/drive/MyDrive/thesis_results/

## What success looks like

- Unit suite: all tests pass.
- Pipeline smoke: several rounds logged, final coverage in the 90s, seconds per round,
  and the loop stopping on its own once refining stops adding coverage.
- Mutation smoke: a mutant count, a score between 0 and 1, and some surviving mutants listed.

Together these show the artifact end to end: it generates, it discards what does not run,
it measures what the survivors cover, it iterates on that signal, and it stops when the
signal flattens.

The numbers here come from one fixture module and are a demonstration, not a result. The
results the thesis reports come from the campaign in `colab_experiments.ipynb`, and the
tables they were reduced to are committed under `experiments/results/analysis/`. To check
those against the thesis without a GPU, run `experiments/verify_reported_numbers.py`.
